# Ensemble tests

Runs every ranker configuration in `CONFIGS` on every dataset in `DATASETS`, evaluates each run against `datasets/<name>/qrels/<QRELS_SPLIT>.tsv`, and collects the metrics in one table.

Vectors must already exist (`python src/encode.py --method <encoder>`). Use the **aws** kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from search import run_search

## Configure

Each entry in `CONFIGS` is `(label, ranker, options)`:
- `"chamfer"` takes `symmetric`.
- `"sw"` (sliced Wasserstein) takes `slices`, `quantiles`, `p`, `seed`. Slices are nested, so for one seed a larger `slices` extends a smaller one with more directions.

`IGNORE_IDENTICAL_IDS` lists datasets whose queries also appear in the corpus (BEIR convention: drop the self-match).

In [ ]:
DATASETS = ["arguana"]
ENCODER = "colbert"
QUERY_TAG = None      # only needed when a dataset has several vector sets for ENCODER
CORPUS_TAG = None
K = 100
QRELS_SPLIT = "test"
METRICS = ("ndcg@10", "recall@100", "mrr@10")
IGNORE_IDENTICAL_IDS = {"arguana"}
WRITE = True          # also save each run to results/<dataset>/<encoder>/

CONFIGS = [
    ("chamfer",     "chamfer", {}),
    ("chamfer-sym", "chamfer", {"symmetric": True}),
] + [
    (f"sw L={L}",   "sw",      {"slices": L, "quantiles": 32, "p": 2, "seed": 0})
    for L in (16, 32, 64, 128, 256, 512)
]

## Run

In [ ]:
rows = []
for dataset in DATASETS:
    for label, ranker, opts in CONFIGS:
        out = run_search(dataset, ranker, ENCODER, QUERY_TAG, CORPUS_TAG, k=K,
                         ignore_identical_ids=dataset in IGNORE_IDENTICAL_IDS, write=WRITE,
                         qrels_split=QRELS_SPLIT, metrics=METRICS, **opts)
        meta = out["meta"]
        if "metrics" not in meta:
            print(f"  no qrels for {dataset}/{QRELS_SPLIT}; skipping evaluation")
        rows.append({"dataset": dataset, "config": label, **meta.get("metrics", {}),
                     "seconds": meta["seconds"]})

## Results

In [ ]:
table = pd.DataFrame(rows)
table

In [ ]:
# Side by side: one row per config, one column group per metric and dataset.
table.pivot(index="config", columns="dataset", values=list(METRICS)).reindex([c[0] for c in CONFIGS])

In [ ]:
# table.to_csv("../results/ensemble.csv", index=False)